In [0]:
# automatically reload all modules before executing new code. The captures changes in local packages.
%load_ext autoreload
%autoreload 2

In [0]:
# If we are using the Databricks wrappers for DSPy or Langchain, we don't need these
api_base = f'https://{spark.conf.get("spark.databricks.workspaceUrl")}/serving-endpoints'
api_key = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

### LangGraph - ReAct Agent

In [0]:
from typing import Annotated, List, Literal, Union

from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command

In [0]:
@tool
def calculator(
  operation: Literal["add", "substract", "multiply", "divide"],
  a: Union[int, float],
  b: Union[int, float],
) -> Union[int, float]:
  """Defines a two-input calculator tool.
  Arg:
      operation (str): The operation to perform ('add', 'subtract', 'multiply', 'divide').
      a (float or int): The first number.
      b (float or int): The second number.
      
  Returns:
      result (float or int): the result of the operation
  Example
      Divide: result   = a / b
      Subtract: result = a - b
  """
  if operation == 'divide' and b == 0:
      return {"error": "Division by zero is not allowed."}

  # Perform calculation
  if operation == 'add':
      result = a + b
  elif operation == 'subtract':
      result = a - b
  elif operation == 'multiply':
      result = a * b
  elif operation == 'divide':
      result = a / b
  else: 
      result = "unknown operation"
  return result

In [0]:
from IPython.display import Image, display
from langgraph.prebuilt import create_react_agent

from databricks_langchain import ChatDatabricks

# Create agent using create_react_agent directly

SYSTEM_PROMPT = "You are a helpful arithmetic assistant who is an expert at using a calculator."

model = ChatDatabricks(
    endpoint = 'databricks-claude-3-7-sonnet',
    extra_params = {"temperature": 0.1}
)

tools = [calculator]

# create agent
agent = create_react_agent(
  model,
  tools,
  prompt=SYSTEM_PROMPT
).with_config({"recursion_limit": 20}) #recursion_limit limits the number of steps the agent will run

# show the agent
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

In [0]:
type(agent)

In [0]:
from utils import format_messages
from langchain_core.output_parsers import JsonOutputParser

# Example usage
# result1 = agent.invoke(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": "What is 3.1 * 4.2? Answer it in JSON format with attribute and result.",
#             }
#         ],
#     }
# )

new_model = model | JsonOutputParser()


result2 = new_model.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is 3.1 * 4.2? Answer it in JSON format with attribute and result.",
            }
        ],
    }
)

In [0]:
result1["messages"][-1].content

In [0]:
format_messages(result1["messages"])

In [0]:
import pprint

pprint.pprint(result1)

In [0]:
len(result1["messages"])

In [0]:
from IPython.display import JSON
from langchain_core.messages import messages_to_dict

JSON({"messages": messages_to_dict(result1["messages"])})

#### Accessing Custom state

In [0]:
from langgraph.prebuilt.chat_agent_executor import AgentState

def reduce_list(left: list | None, right: list | None) -> list:
    """Safely combine two lists, handling cases where either or both inputs might be None.

    Args:
        left (list | None): The first list to combine, or None.
        right (list | None): The second list to combine, or None.

    Returns:
        list: A new list containing all elements from both input lists.
               If an input is None, it's treated as an empty list.
    """
    if not left:
        left = []
    if not right:
        right = []
    return left + right

class CalcState(AgentState):
    """Graph State."""
    ops: Annotated[List[str], reduce_list]

In [0]:
@tool
def calculator_wstate(
  operation: Literal["add", "substract", "multiply", "divide"],
  a: Union[int, float],
  b: Union[int, float],
  state: Annotated[CalcState, InjectedState], # not sent to LLM,
  tool_call_id: Annotated[str, InjectedToolCallId] # not sent to LLM
) -> Union[int, float]:
  """Defines a two-input calculator tool.

  Arg:
      operation (str): The operation to perform ('add', 'subtract', 'multiply', 'divide').
      a (float or int): The first number.
      b (float or int): The second number.
      
  Returns:
      result (float or int): the result of the operation
  Example
      Divide: result   = a / b
      Subtract: result = a - b
  """
  if operation == 'divide' and b == 0:
      return {"error": "Division by zero is not allowed."}

  # Perform calculation
  if operation == 'add':
      result = a + b
  elif operation == 'subtract':
      result = a - b
  elif operation == 'multiply':
      result = a * b
  elif operation == 'divide':
      result = a / b
  else: 
      result = "unknown operation"

  ops = [f"({operation}, {a}, {b}),"]

  return Command(
    update={
      "ops": ops,
      "messages": [
        ToolMessage(f"{result}", tool_call_id=tool_call_id)
      ]
    }
  )


In [0]:
SYSTEM_PROMPT = "You are a helpful arithmetic assistant who is an expert at using a calculator."

tools = [calculator_wstate] # new tool with access to update state

# Create Agent
agent = create_react_agent(
  model,
  tools,
  prompt=SYSTEM_PROMPT,
  state_schema=CalcState, # now defining state schema
).with_config({"recursion_limit": 20}) #recursion_limit limits the number of steps the agent will run



In [0]:
# Example usage
result2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is 3.1 * 4.2?",
            }
        ],
    }
)

format_messages(result2["messages"])

In [0]:
pprint.pprint(result2)

In [0]:
# Example usage
result3 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is 3.1 * 4.2 + 5.5 * 6.5?",
            }
        ],
    }
)

format_messages(result3["messages"])

In [0]:
import json
print(json.dumps(result3, indent=4, separators=(',', ':'), default=str))

In [0]:
pprint.pprint(result3)

In [0]:
# Example usage - create your own
result4 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Create an example of your own?",
            }
        ],
    }
)

format_messages(result4["messages"])

## DSPy - ReAct Agent

In [0]:
import dspy
import databricks_dspy
from databricks_dspy import DatabricksLM, DatabricksRM

In [0]:
dspy.configure(lm=DatabricksLM(model="databricks/databricks-claude-3-7-sonnet"))

predict = dspy.Predict("question->answer")
print(predict(question="why did a chicken cross the kitchen?"))